# Fashion MNIST Multinomial

In this notebook we want to change the code from binary classification to multinomial classificationGet the necessary files for the next session from https://www.kaggle.com/datasets/zalando-research/fashionmnist


## Preparing the data

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
print(tf.__version__)

In [ ]:
#read data with pandas
dataTrain = pd.read_csv('data/fashion-mnist_train.csv')
dataTest = pd.read_csv('data/fashion-mnist_test.csv')

#convert to numpy and initialize variables for training
X_train = dataTrain.iloc[:, 1:].to_numpy()  # alle spalten 1..n in Features
y_train = dataTrain.iloc[:, 0].to_numpy()   # reste spalte 0 in den Lables-Vektor
X_test = dataTest.iloc[:, 1:].to_numpy()
y_test = dataTest.iloc[:, 0].to_numpy()

#Define Classnames
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print("Training Features: ", X_train.shape)
print("Test Features:     ",X_test.shape)
print("Training Labels:   ",y_train.shape)
print("Test Labels:       ",y_test.shape)

# Modifying the Network for multinomial classification

In [ ]:
# Old Model
model = Sequential()
model.add(Input((784,)))
model.add(Dense(100, activation="sigmoid", name="inputlayer"))
model.add(Dense(1, activation="sigmoid"))
model.compile(optimizer="sgd", loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
# New adapted model 

<details>
  <summary><i>Solution</i></summary>
  <code>
model = Sequential()
model.add(Input((784,)))
model.add(Dense(100, activation="sigmoid"))
model.add(Dense(10, activation="softmax"))

#anything necessary to change here 
model.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])
  </code>  
</details>

In [ ]:
model.summary()

In [ ]:
# show the that string values given here correspond to classes e.g tf.nn.softmax or 
print(model.layers[0].activation)
print(model.layers[1].activation)


# Prepare to train the network

In [ ]:
# now we work with all labels
y_train

In [ ]:
# prepare ground truth as one-hot encoded values
y_one_hot = tf.one_hot(y_train,len(class_names))
y_one_hot_test = tf.one_hot(y_test,len(class_names))

In [ ]:
# look at two examples
y_one_hot[0:2]

In [ ]:
#print the shapes
print("Shape of training target values", y_one_hot.shape)
print("Shape of test target values", y_one_hot_test.shape)

In [ ]:
tf.one_hot(y_train, len(class_names))

In [ ]:
#alternative to tf.one_hot 
tf.keras.utils.to_categorical(y_train, num_classes=len(class_names))

In [ ]:
# renew model
model = Sequential()
model.add(Input((784,)))
model.add(Dense(100, activation="sigmoid"))
model.add(Dense(10, activation="softmax"))

model.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
model.fit(
    X_train,
    y_one_hot,
    epochs=10,
    batch_size=100)

# Evaluate and use

In [ ]:
model.evaluate(X_test, y_one_hot_test)

In [ ]:
y_pred = model.predict(X_test)
print(f"Shape: {y_pred.shape}")
y_pred

# Convert probabilities to class 

In [ ]:
# introducing argmax
pred = y_pred[1]
print(np.round(pred*100))
index1 = np.argmax(pred) 
index1

In [ ]:
# argmax for the whole array
index = tf.argmax(y_pred, axis=1) # alternativ auch np.argmax()
print(index[0:4])

In [ ]:
# depending on your current installation you might need the follwing :
#!conda install scikit-learn

In [ ]:
#instead of using model.evaluate you can also use sklearn functions to determine accuracy
from sklearn.metrics import accuracy_score   
score = accuracy_score(y_test, index)  
score

# Why are we not as good as before ?
Previously we had 95% accurracy for binary classification

10% TShirts 90% Others

In [ ]:
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
print(f"Precision: {precision_score(y_test, index, average='weighted')}")  
print(f"Recall: {recall_score(y_test, index, average='weighted')}")  


# What is being mixed up ?

## Creating a confusion matrix for better insight 

In [ ]:
import pandas as pd

ytrue = pd.Series(y_test, name = 'actual')
ypred = pd.Series(index, name = 'pred')
pd.crosstab(ytrue, ypred)

The way we learned it last year:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm= confusion_matrix(y_test, index)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(xticks_rotation='vertical')


# Exercises

* Try different activation functions in the first layer

* Try what happens if we use sigmoid for the last layer ? Does categorical-crossentropy work together with sigmoid as well ?

* Try what happens if we have a relu in the last layer + categorical_hinge ?

